In [2]:
# Cell 1: Env setup
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("✅ Env set")

✅ Env set


In [3]:
%%capture
!pip install torch==2.4.0 torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.45.2 tokenizers==0.20.3
!pip install accelerate==0.34.0 bitsandbytes==0.43.0
!pip install peft==0.14.0 datasets pillow
!pip install jiwer einops addict easydict huggingface_hub

In [4]:
import torch
print(f"Pytorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Pytorch: 2.4.0+cu121
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [5]:
# Cell 4: Download base model
import sys
from huggingface_hub import snapshot_download

MODEL_DIR = "./deepseek_ocr2"
snapshot_download("deepseek-ai/DeepSeek-OCR-2", local_dir=MODEL_DIR)
if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)
print("✅ Base model downloaded")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Base model downloaded


In [6]:
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel
import torch

HF_TOKEN = "YOUR_HF_TOKEN"

base_model = AutoModel.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    use_safetensors=True,
)
base_model = base_model.to("cuda:0")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    use_fast=True,
)

# LoRA weights HF se load karo
model = PeftModel.from_pretrained(
    base_model,
    "zahidazam714/khowar-ocr-v0.1",
    torch_dtype=torch.bfloat16,
    is_trainable=True,  # ✅ Training ke liye
)

print("✅ Model loaded from HF checkpoint")
print(f"Device: {next(model.parameters()).device}")
model.print_trainable_parameters()

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


✅ Model loaded from HF checkpoint
Device: cuda:0
trainable params: 86,307,840 || all params: 3,475,427,200 || trainable%: 2.4834


In [ ]:
# Cell 6: Patch masked_scatter_ issue
import os
file_path = os.path.expanduser(
    "~/.cache/huggingface/modules/transformers_modules/deepseek_ocr2/modeling_deepseekocr2.py"
)
with open(file_path, 'r') as f:
    content = f.read()
if 'inputs_embeds = inputs_embeds.clone()  # patched' not in content:
    content = content.replace(
        'inputs_embeds[idx].masked_scatter_(',
        'inputs_embeds = inputs_embeds.clone()  # patched\n        inputs_embeds[idx].masked_scatter_('
    )
    with open(file_path, 'w') as f:
        f.write(content)
    print("✅ Patched")
else:
    print("✅ Already patched")

✅ Patched


In [8]:
from huggingface_hub import login
login(token=HF_TOKEN)
print("✅ HF logged in")

✅ HF logged in


In [9]:
# Cell 7: Load dataset
from datasets import load_dataset
import os

DATASET_DIR = "/kaggle/input/datasets/zahidazam/deepseek-ocr-2-finetuning-dataset"
dataset = load_dataset("json", data_files={
    "train": f"{DATASET_DIR}/train.jsonl",
    "validation": f"{DATASET_DIR}/val.jsonl",
})
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])}")

Train: 15971 | Val: 1996


In [10]:
# Cell 8: Convert dataset
def convert_user_format(sample):
    conversation = []
    for msg in sample["messages"]:
        if msg["role"] == "user":
            text_content = None
            img_path = None
            for part in msg["content"]:
                if part["type"] == "image":
                    img_path = os.path.join(DATASET_DIR, part["image"])
                elif part["type"] == "text":
                    text_content = part["text"]
            conversation.append({
                "role": "<|User|>",
                "content": text_content,
                "image_paths": [img_path]
            })
        elif msg["role"] == "assistant":
            conversation.append({
                "role": "<|Assistant|>",
                "content": msg["content"]
            })
    return {"messages": conversation}

converted_train = [convert_user_format(s) for s in dataset["train"]]
converted_val = [convert_user_format(s) for s in dataset["validation"]]
print(f"✅ Converted: {len(converted_train)} train, {len(converted_val)} val")

✅ Converted: 15971 train, 1996 val


In [11]:
# Cell 9: Define Collator
import torch, math, io
from dataclasses import dataclass
from typing import Any
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
from deepseek_ocr2.modeling_deepseekocr2 import text_encode, BasicImageTransform, dynamic_preprocess

@dataclass
class DeepSeekOCR2DataCollator:
    tokenizer: Any
    model: Any

    def __init__(self, tokenizer, model, image_size=768, base_size=1024,
                 crop_mode=True, train_on_responses_only=True):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.train_on_responses_only = train_on_responses_only
        self.dtype = next(model.parameters()).dtype
        self.image_transform = BasicImageTransform(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5), normalize=True)
        self.patch_size = 16
        self.downsample_ratio = 4
        self.bos_id = tokenizer.bos_token_id or 0

    def load_image(self, src):
        if isinstance(src, str):
            return Image.open(src).convert("RGB")
        elif isinstance(src, Image.Image):
            return src.convert("RGB")
        elif isinstance(src, dict) and 'bytes' in src:
            return Image.open(io.BytesIO(src['bytes'])).convert("RGB")
        raise ValueError(f"Unknown image type: {type(src)}")

    def process_image(self, image):
        images_list, images_crop_list, images_spatial_crop = [], [], []
        if self.crop_mode:
            if image.size[0] <= 768 and image.size[1] <= 768:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num=2, max_num=6, image_size=self.image_size, use_thumbnail=False)
            global_view = ImageOps.pad(image, (self.base_size, self.base_size),
                color=tuple(int(x*255) for x in self.image_transform.mean))
            images_list.append(self.image_transform(global_view).to(self.dtype))
            w, h = crop_ratio
            images_spatial_crop.append([w, h])
            if w > 1 or h > 1:
                for c in images_crop_raw:
                    images_crop_list.append(self.image_transform(c).to(self.dtype))
            nq = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            nqb = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tok = ([self.image_token_id] * nqb) * nqb + [self.image_token_id]
            if w > 1 or h > 1:
                tok += ([self.image_token_id] * (nq * w)) * (nq * h)
        else:
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])
            global_view = ImageOps.pad(image, (self.base_size, self.base_size),
                color=tuple(int(x*255) for x in self.image_transform.mean))
            images_list.append(self.image_transform(global_view).to(self.dtype))
            nq = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tok = ([self.image_token_id] * nq) * nq + [self.image_token_id]
        return images_list, images_crop_list, images_spatial_crop, tok, crop_ratio

    def process_single_sample(self, messages):
        images = []
        for msg in messages:
            for src in msg.get("image_paths", []) + msg.get("images", []):
                if src is not None:
                    images.append(self.load_image(src))
        if not images:
            raise ValueError("No images found")
        tokenized_str, images_seq_mask = [self.bos_id], [False]
        images_list, images_crop_list, images_spatial_crop = [], [], []
        prompt_token_count = -1
        assistant_started = False
        image_idx = 0
        for msg in messages:
            role = msg["role"]
            content = msg["content"]
            if role == "<|Assistant|>":
                if not assistant_started:
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True
                content = f"{content.strip()} {self.tokenizer.eos_token}"
            parts = content.split('<image>')
            for i, part in enumerate(parts):
                toks = text_encode(self.tokenizer, part, bos=False, eos=False)
                tokenized_str.extend(toks)
                images_seq_mask.extend([False] * len(toks))
                if i < len(parts) - 1:
                    img = images[image_idx]
                    il, cl, sc, ti, _ = self.process_image(img)
                    images_list.extend(il)
                    images_crop_list.extend(cl)
                    images_spatial_crop.extend(sc)
                    tokenized_str.extend(ti)
                    images_seq_mask.extend([True] * len(ti))
                    image_idx += 1
        if not assistant_started:
            prompt_token_count = len(tokenized_str)
        images_ori = torch.stack(images_list)
        images_spatial_crop_t = torch.tensor(images_spatial_crop, dtype=torch.long)
        images_crop = torch.stack(images_crop_list) if images_crop_list else \
            torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)
        return {
            "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": images_spatial_crop_t,
            "prompt_token_count": prompt_token_count,
        }

    def __call__(self, features):
        batch_data = []
        for f in features:
            try:
                batch_data.append(self.process_single_sample(f['messages']))
            except Exception as e:
                print(f"Skipping sample: {e}")
        if not batch_data:
            raise ValueError("Empty batch")
        input_ids = pad_sequence([x['input_ids'] for x in batch_data],
                                  batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence([x['images_seq_mask'] for x in batch_data],
                                        batch_first=True, padding_value=False)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        labels[images_seq_mask] = -100
        if self.train_on_responses_only:
            for i, x in enumerate(batch_data):
                pc = x['prompt_token_count']
                if pc > 0:
                    labels[i, :pc] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": (input_ids != self.tokenizer.pad_token_id).long(),
            "labels": labels,
            "images": [
                (x['images_crop'].to("cuda:0"), x['images_ori'].to("cuda:0"))
                for x in batch_data
            ],
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": torch.cat([x['images_spatial_crop'] for x in batch_data]),
        }

data_collator = DeepSeekOCR2DataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=768,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=True,
)
print("✅ Collator ready")

✅ Collator ready


In [12]:
# Cell 10: Filter noisy prints
import builtins
_real_print = builtins.print

def _filtered_print(*args, **kwargs):
    msg = " ".join(str(a) for a in args)
    if "BASE:" in msg or "PATCHES:" in msg or "======" in msg:
        return
    _real_print(*args, **kwargs)

builtins.print = _filtered_print
print("✅ Noisy prints filtered")

✅ Noisy prints filtered


In [13]:
# Cell 11: Resume Training — 400 more steps, auto-push to HF
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/kaggle/working/khowar_ocr_outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    max_steps=400,              # 400 more steps
    learning_rate=1e-4,         # ✅ Thoda kam — fine-tuning pe conservative
    warmup_ratio=0.05,
    weight_decay=0.001,
    logging_steps=10,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=False,
    lr_scheduler_type="cosine",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none",
    local_rank=-1,
    # ✅ Auto push to HF — session expire hone pe bhi safe
    push_to_hub=True,
    hub_model_id="zahidazam714/khowar-ocr-v0.1",
    hub_strategy="checkpoint",
    hub_token=HF_TOKEN,
)

model.train()

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=converted_train,
    args=training_args,
)

gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU = {gpu_stats.name} | VRAM = {gpu_stats.total_memory / 1e9:.1f} GB")
print(f"Reserved = {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(f"Max steps = 400 | LR = 1e-4 | Auto-push to HF = ON")

trainer_stats = trainer.train()

# ✅ Auto final push after training complete
trainer.push_to_hub(commit_message="khowar-ocr-v0.1 — step 800 final")
print("✅ Final model pushed to HF!")

2026-05-12 10:32:08.650962: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778581928.670717    1212 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778581928.676278    1212 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778581928.691208    1212 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778581928.691222    1212 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778581928.691225    1212 computation_placer.cc:177] computation placer alr

GPU = Tesla T4 | VRAM = 15.6 GB
Reserved = 7.48 GB
Max steps = 400 | LR = 1e-4 | Auto-push to HF = ON


The attention layers in this model are transitioning from computing the RoPE embeddings internally through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed `position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be removed and `position_embeddings` will be mandatory.


Step,Training Loss
10,0.950000
20,0.966900
30,0.636900
40,0.655500
50,0.687800
60,0.672900
70,0.652600
80,0.607700
90,0.680300
100,0.713700


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


ValueError: Invalid metadata in README.md.
- "base_model" with value "./deepseek_ocr2" is not valid. Use a model id from https://hf.co/models.